# Medicine pronunciation dictionary for Sarvam text to speech

A clinic app that reads a prescription aloud has to get the drug name right. Drug names
are coined words, not dictionary words, so a speech model has nothing to fall back on.
On top of that an Indian prescription is written in shorthand -- `BD`, `TDS`, `1-0-1` --
which a speech engine reads out as letters and hyphens rather than as instructions.

Sarvam's `bulbul:v3` has a feature built for exactly this: a **pronunciation dictionary**,
a JSON file of word-to-replacement pairs the engine applies before synthesis.

**This notebook has not been executed.** There was no Sarvam api key on the machine it
was written on, so every code cell below ships with an empty output and nothing here has
ever been run against the live API. Run it yourself with your own key and check what
comes back before you rely on any of it.

**This is a pronunciation and text-to-speech recipe, not a medical tool.** It gives no
dosing guidance, no interaction checking, no substitution suggestions and no clinical
advice. The shorthand expansion is a *reading* of what the prescriber wrote. It never
computes, recommends, adjusts or validates a dose.

What happens below:

1. Read the prescription lines and show the reading of every shorthand on them.
2. Validate the dictionary file offline -- language keys, word cap, file size.
3. Screen the drug list for look-alike names.
4. Show the exact text the engine will receive, per language.
5. Upload the dictionary, read it back, synthesise with and without it, then delete it.

Steps 1 to 4 need no key at all and run against `pronunciation.py`, which imports
nothing outside the standard library. Only step 5 onward talks to the API.

In [ ]:
%pip install -q "sarvamai>=0.1.24" "python-dotenv>=1.0.0"

## 1. Setup

The api key is passed to `SarvamAI` explicitly. Its default argument is evaluated once
when the SDK is imported, so a key set after that import is never picked up and the
client raises instead.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from sarvamai import SarvamAI
from sarvamai.play import save

RECIPE_DIR = Path.cwd()
if not (RECIPE_DIR / "pronunciation.py").exists():
    raise RuntimeError(
        "Run this notebook from examples/medicine-pronunciation-dictionary/ so that "
        "pronunciation.py and medicine_pronunciation.json sit next to it."
    )
sys.path.insert(0, str(RECIPE_DIR))

DICTIONARY_PATH = RECIPE_DIR / "medicine_pronunciation.json"
OUTPUT_DIR = RECIPE_DIR / "outputs"

load_dotenv()
if not os.environ.get("SARVAM_API_KEY"):
    raise RuntimeError("Set SARVAM_API_KEY in your environment or in a .env file first.")

client = SarvamAI(api_subscription_key=os.environ["SARVAM_API_KEY"])

## 2. The prescription lines

**These lines are invented for this recipe.** They are not extracted from any real
prescription, any dataset or any patient record. `render_transcript` prints each line
exactly as written and puts the reading of every shorthand on it in brackets beside it,
so a human can check the reading against the paper.

In [ ]:
import pronunciation as pron

PRESCRIPTION_LINES = [
    "Tab Amlodipine 5 mg OD",
    "Tab Amiloride 5 mg 1-0-0",
    "Tab Metformin 500 mg 1-0-1 after food",
    "Tab Paracetamol 500 mg SOS",
    "Cap Omeprazole 20 mg HS",
    "Tab Atenolol 25 mg BD",
    "Syp Amoxicillin 5 ml TDS",
    "Tab Prednisolone 10 mg 2-1-1 after food",
]

print(pron.render_transcript("\n".join(PRESCRIPTION_LINES)))

## 3. Validate the dictionary offline

Nothing in the SDK checks any of this. A dictionary's language keys are typed as plain
strings, so a typo such as `hi_IN`, or a code that is valid for speech to text but not
for speech, is accepted without complaint and then matches nothing -- the upload looks
fine and the dictionary silently does nothing. The 100-word and 1 MB caps live in a
docstring and are not counted or measured before the upload either.

The word cap is read here as a **total across all blocks**, not a per-block count. Sarvam
says "words per dictionary", which reads as the total; this file budgets to 90 so a wrong
reading of that sentence is not what breaks the recipe.

In [ ]:
dictionary = pron.load_dictionary(DICTIONARY_PATH)
findings = pron.validate_dictionary(DICTIONARY_PATH)

for finding in findings:
    print(f"{finding.check}: {finding.message}")

blocks = dictionary["pronunciations"]
print("findings:", len(findings))
print("blocks:", sorted(blocks))
print("entries per block:", {code: len(block) for code, block in blocks.items()})
print("total entries:", sum(len(block) for block in blocks.values()), "of", pron.MAX_WORDS)
print("file size:", DICTIONARY_PATH.stat().st_size, "of", pron.MAX_FILE_BYTES, "bytes")

## 4. Screen the drug list for look-alike names

Two signals, either sufficient: a sequence-match ratio at or above 0.70, or a shared
head and tail with a small length difference. The second limb is what catches pairs the
first misses -- a hurried reader takes in the start and the end of a coined word and
fills in the middle.

This is a prompt to check a list by eye, **not a safety system**. A string metric cannot
recover names that are confused because of packaging, shelf position or handwriting, and
the README names the pairs this rule is known to miss. No pair table is shipped anywhere
in this recipe: every pair below is derived from the word list by the rule.

In [ ]:
drug_names = sorted(set(blocks["en-IN"]) - set(pron.SHORTHAND_EXPANSIONS))

for pair in pron.find_confusable_pairs(drug_names):
    print(f"{pair.a:<14}{pair.b:<14}score={pair.score:.3f}  rule={pair.rule}")

## 5. The exact text the engine will receive

Dose patterns are expanded first, by code, because there are 125 single-digit forms and
they will not fit in a 100-word dictionary. Then the dictionary substitution is applied
per language, offline. Matching here is assumed to be whole-word and case-sensitive:
Sarvam documents neither, so this preview is an approximation of what the engine does,
not a copy of it.

In [ ]:
for line in PRESCRIPTION_LINES[:4]:
    spoken = pron.expand_dose_pattern(line)
    print("written:", line)
    for code in ("hi-IN", "ta-IN", "en-IN"):
        print(f"  {code}:", pron.apply_dictionary(spoken, dictionary, code))
    print()

## 6. Upload the dictionary

The file goes up as an explicit three-tuple, `(filename, bytes, content type)`. This
endpoint applies no default content type anywhere in the SDK, so a bare file handle
leaves both the multipart filename and the content type to inference.

An account holds at most 10 dictionaries, so the count is printed before the upload and
again after the delete at the end.

In [ ]:
dictionary_bytes = DICTIONARY_PATH.read_bytes()

before = client.pronunciation_dictionary.list()
print("dictionaries on the account before upload:", before.dictionary_count)

created = client.pronunciation_dictionary.create(
    file=("dictionary.json", dictionary_bytes, "application/json"),
)
dict_id = created.dictionary_id
print("dictionary_id:", dict_id)

## 7. Read it back

`create` returns an id and nothing else that can be relied on -- it does not echo the
uploaded content. So the only honest way to prove the upload landed is to fetch it.

Every response model allows unknown fields, so only the documented ones are read here.
Nothing iterates a response as if its shape were closed.

In [ ]:
fetched = client.pronunciation_dictionary.get(dict_id)

print("blocks on the server:", sorted(fetched.pronunciations))
print("entries on the server:", sum(len(block) for block in fetched.pronunciations.values()))
print("hi-IN reading of Amlodipine:", fetched.pronunciations["hi-IN"]["Amlodipine"])

## 8. Synthesise, with and without the dictionary

`model="bulbul:v3"` is passed on both calls. Leaving `model` out does not merely produce
worse audio: the server falls back to an older model that ignores `dict_id` entirely, so
the dictionary would be silently skipped, which is the whole point of the recipe.

Note also that the parameter is `language_code`, not `target_language_code`, and that
`shubh` is a speaker the current model actually supports.

We cannot hear the result from here, so this recipe makes no claim about the audio. What
it does show is the text the engine receives, in step 5.

In [ ]:
SPOKEN_LINE = pron.expand_dose_pattern(PRESCRIPTION_LINES[2])
print("sent to the engine:", SPOKEN_LINE)

without = client.text_to_speech.convert(
    text=SPOKEN_LINE,
    model="bulbul:v3",
    language_code="hi-IN",
    speaker="shubh",
)
save(without, str(OUTPUT_DIR / "without_dictionary.wav"))

with_dict = client.text_to_speech.convert(
    text=SPOKEN_LINE,
    model="bulbul:v3",
    language_code="hi-IN",
    speaker="shubh",
    dict_id=dict_id,
)
save(with_dict, str(OUTPUT_DIR / "with_dictionary.wav"))

print("wrote both files to", OUTPUT_DIR)

## 9. Update the dictionary in place

`update` takes `dict_id` as a keyword argument and re-uploads the whole file. `get` is
the one call in this family that takes `dict_id` positionally.

In [ ]:
updated = client.pronunciation_dictionary.update(
    dict_id=dict_id,
    file=("dictionary.json", dictionary_bytes, "application/json"),
)
print("updated dictionary_id:", updated.dictionary_id)
print("blocks after the update:", sorted(updated.updated_pronunciations))

## 10. Delete it again

Ten dictionaries per account is a small budget. This cleans up after itself and prints
the count again so you can see the slot was given back.

In [ ]:
removed = client.pronunciation_dictionary.delete(dict_id=dict_id)
print("delete succeeded:", removed.success)
print("server said:", removed.message)

after = client.pronunciation_dictionary.list()
print("dictionaries on the account after delete:", after.dictionary_count)

## What this recipe does not do

- No general drug database. Ninety entries total, curated by hand.
- No clinical advice of any kind. See the note at the top.
- No streaming or WebSocket synthesis. `dict_id` works there too, but it could not be
  run here, so it is not shown.
- No claim about audio quality. Nothing here was heard.
- No languages beyond `hi-IN`, `ta-IN` and `en-IN`. Matching is per language block, so a
  fourth language costs thirty entries against the cap for no extra demonstration.

Swap in your own word list by editing `medicine_pronunciation.json` and re-running steps
3 and 4. The validator runs on any file you give it, so the whole offline half of this
recipe is useful before you obtain a key.